# GeoAI Land Cover Classifier — 02 · Preparación de datos

**Dataset:** EuroSAT RGB (Sentinel-2) · **Entorno:** Google Colab · **Estado:** experimentos exploratorios.

**Ejecución:** `Entorno de ejecución → Ejecutar todo` desde una sesión nueva. Cada notebook prepara las variables que necesita; no depende de otro notebook abierto.

**Datos:** `./data` se descarga temporalmente en Colab. **Nunca utilices el conjunto test para escoger modelos.**

**Reproducibilidad:** `random_state=42`; pueden existir diferencias de tiempo o pequeñas variaciones entre versiones de dependencias.


In [ ]:

import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import PIL

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Pillow:", PIL.__version__)


In [ ]:

from torchvision.datasets import EuroSAT

dataset = EuroSAT(
    root="./data",
    download=True
)

print("Número de imágenes:", len(dataset))
print("Categorías:", dataset.classes)

imagen, etiqueta = dataset[0]

print("Tamaño de la imagen:", imagen.size)
print("Primera categoría:", dataset.classes[etiqueta])



## 04. División del dataset

### Objetivo
Separar EuroSAT en tres conjuntos independientes:

- Entrenamiento: 70 %
- Validación: 15 %
- Prueba: 15 %

### Conceptos importantes

**Train:** datos utilizados para ajustar el modelo.

**Validation:** datos utilizados para comparar modelos,
seleccionar hiperparámetros y tomar decisiones.

**Test:** datos reservados para medir el rendimiento final.

**Estratificación:** técnica que mantiene aproximadamente
la misma distribución de clases en cada conjunto.

**Data leakage:** ocurre cuando información que debería
estar reservada para la evaluación influye en el entrenamiento
o en la selección del modelo.

### Precaución geoespacial

Esta división se realiza por imagen y etiqueta, no por
ubicación geográfica.

Si existen imágenes próximas o espacialmente relacionadas,
los resultados podrían sobreestimar la capacidad de
generalización a regiones completamente nuevas.

Por tanto, interpretaremos las métricas como resultados
sobre una división aleatoria de EuroSAT, no como evidencia
de generalización geográfica independiente.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 04: División estratificada del dataset
# ============================================================

# train_test_split divide una colección de datos en
# subconjuntos aleatorios, de forma reproducible.
from sklearn.model_selection import train_test_split

# NumPy nos permite trabajar con arrays numéricos.
import numpy as np

# ------------------------------------------------------------
# 1. Preparar índices y etiquetas
# ------------------------------------------------------------

# Cada índice identifica una imagen del dataset.
# Por ejemplo, el índice 0 corresponde a dataset[0].
#
# No cargamos todas las imágenes en memoria:
# trabajamos únicamente con sus posiciones.
indices = np.arange(len(dataset))

# Convertimos las etiquetas a un array de NumPy.
etiquetas = np.array(dataset.targets)

# ------------------------------------------------------------
# 2. Primera división: Train (70 %) y temporal (30 %)
# ------------------------------------------------------------

# test_size=0.30 reserva el 30 % para dividirlo después
# entre validación y prueba.
#
# random_state=42 fija la semilla aleatoria y permite
# reproducir exactamente esta división.
#
# stratify=etiquetas conserva las proporciones de clases.
indices_train, indices_temporales = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=etiquetas
)

# ------------------------------------------------------------
# 3. Segunda división: Validation (15 %) y Test (15 %)
# ------------------------------------------------------------

# Obtenemos las etiquetas de las imágenes temporales.
etiquetas_temporales = etiquetas[indices_temporales]

# Dividimos el 30 % reservado en dos mitades iguales.
#
# El 50 % del 30 % original equivale al 15 % del total.
indices_val, indices_test = train_test_split(
    indices_temporales,
    test_size=0.50,
    random_state=42,
    stratify=etiquetas_temporales
)

# ------------------------------------------------------------
# 4. Mostrar los tamaños
# ------------------------------------------------------------

print("Entrenamiento:", len(indices_train))
print("Validación:", len(indices_val))
print("Prueba:", len(indices_test))

# Verificamos que no falte ninguna imagen.
assert (
    len(indices_train)
    + len(indices_val)
    + len(indices_test)
    == len(dataset)
), "Error: el número de imágenes no coincide."

# Comprobamos que ningún índice esté repetido
# entre los tres conjuntos.
assert len(
    set(indices_train)
    | set(indices_val)
    | set(indices_test)
) == len(dataset), "Error: hay índices duplicados."

print("División completada correctamente.")



## 05. Verificación de la estratificación

### Objetivo

Comprobar que entrenamiento, validación y prueba
conservan aproximadamente la distribución original
de las diez clases de EuroSAT.

### Conceptos

- **Estratificación:** reparto de los datos manteniendo
  aproximadamente la proporción de cada clase.

- **Distribución relativa:** porcentaje de imágenes
  que pertenece a cada categoría.

- **Validación del split:** comprobación de que los
  conjuntos se han construido como esperamos.

### ¿Qué esperamos?

Que la proporción de cada categoría sea muy similar
en los tres conjuntos.

Esta comprobación no garantiza independencia
geográfica entre imágenes.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 05: Verificación de la estratificación
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Crear una función para calcular proporciones
# ------------------------------------------------------------

def calcular_distribucion(indices_conjunto):
    """
    Calcula el porcentaje de imágenes de cada clase.

    Parámetros
    ----------
    indices_conjunto:
        Índices de las imágenes que pertenecen al conjunto.

    Devuelve
    --------
    Un array con los porcentajes de las 10 categorías,
    en el mismo orden que dataset.classes.
    """

    # Recuperamos las etiquetas correspondientes
    # a las imágenes de este conjunto.
    etiquetas_conjunto = etiquetas[indices_conjunto]

    # Contamos cuántas imágenes hay de cada clase.
    # minlength garantiza que el resultado incluya
    # las 10 categorías, incluso si alguna tuviera 0.
    conteos = np.bincount(
        etiquetas_conjunto,
        minlength=len(dataset.classes)
    )

    # Convertimos los recuentos a porcentajes.
    porcentajes = conteos / len(indices_conjunto) * 100

    return porcentajes


# ------------------------------------------------------------
# 2. Calcular la distribución de cada conjunto
# ------------------------------------------------------------

distribucion_train = calcular_distribucion(indices_train)
distribucion_val = calcular_distribucion(indices_val)
distribucion_test = calcular_distribucion(indices_test)

# ------------------------------------------------------------
# 3. Crear una tabla comparativa
# ------------------------------------------------------------

tabla_estratificacion = pd.DataFrame({
    "Categoría": dataset.classes,
    "Train (%)": distribucion_train,
    "Validation (%)": distribucion_val,
    "Test (%)": distribucion_test
})

# Redondeamos únicamente para mostrar los resultados.
# Los arrays originales conservan su precisión.
display(tabla_estratificacion.round(2))

# ------------------------------------------------------------
# 4. Medir la diferencia máxima entre conjuntos
# ------------------------------------------------------------

# Calculamos, por cada categoría, la diferencia entre
# el porcentaje más alto y el más bajo.
columnas_porcentajes = [
    "Train (%)",
    "Validation (%)",
    "Test (%)"
]

diferencias = (
    tabla_estratificacion[columnas_porcentajes].max(axis=1)
    - tabla_estratificacion[columnas_porcentajes].min(axis=1)
)

print(
    "Diferencia máxima entre conjuntos:",
    round(diferencias.max(), 3),
    "puntos porcentuales"
)

# ------------------------------------------------------------
# 5. Visualizar los porcentajes
# ------------------------------------------------------------

# Cada categoría tendrá tres barras:
# una de entrenamiento, otra de validación y otra de prueba.
eje = tabla_estratificacion.set_index("Categoría").plot(
    kind="bar",
    figsize=(13, 5),
    width=0.8
)

eje.set_title("EuroSAT — Distribución por conjunto")
eje.set_xlabel("Categoría")
eje.set_ylabel("Porcentaje de imágenes (%)")

# Ajustamos las etiquetas del eje horizontal.
plt.xticks(rotation=45, ha="right")

# Evitamos que los elementos se superpongan.
plt.tight_layout()

# Mostramos el gráfico.
plt.show()



## 06. Preparación de imágenes para Machine Learning

### Objetivo

Convertir las imágenes RGB de EuroSAT en vectores numéricos
para utilizarlos como entrada de un modelo de scikit-learn.

### Conceptos

- **Píxel:** unidad mínima de información de una imagen.
- **RGB:** representación mediante tres canales de color:
  rojo, verde y azul.
- **Feature:** característica numérica utilizada como
  entrada por un modelo.
- **Flattening:** transformación de una matriz
  multidimensional en un vector unidimensional.
- **Normalización:** transformación de los valores a
  una escala determinada.

### Decisión técnica

Usaremos inicialmente los valores RGB normalizados
entre 0 y 1.

Esta representación servirá para crear un modelo
de referencia, aunque no aprovecha explícitamente
las relaciones espaciales entre los píxeles.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 06: Conversión de imágenes a características
# ============================================================

import numpy as np

# ------------------------------------------------------------
# 1. Seleccionar una imagen del conjunto de entrenamiento
# ------------------------------------------------------------

# Usamos el primer índice de entrenamiento.
# No utilizamos el conjunto de prueba para experimentar.
indice_imagen = indices_train[0]

# Recuperamos la imagen y su etiqueta.
imagen_pil, etiqueta = dataset[indice_imagen]

# ------------------------------------------------------------
# 2. Convertir la imagen a un array numérico
# ------------------------------------------------------------

# Una imagen PIL es un objeto preparado para trabajar
# con imágenes. NumPy permite representar sus píxeles
# como números.
imagen_array = np.array(imagen_pil)

print("Forma original:", imagen_array.shape)
print("Tipo de dato:", imagen_array.dtype)

# ------------------------------------------------------------
# 3. Normalizar los valores RGB
# ------------------------------------------------------------

# Las imágenes RGB de 8 bits utilizan valores entre
# 0 y 255 para representar la intensidad de cada canal.
#
# Convertimos los valores a float32 y dividimos entre
# 255 para obtener valores entre 0 y 1.
imagen_normalizada = imagen_array.astype(np.float32) / 255.0

print("Valor mínimo:", imagen_normalizada.min())
print("Valor máximo:", imagen_normalizada.max())

# ------------------------------------------------------------
# 4. Convertir la imagen en un vector
# ------------------------------------------------------------

# flatten() transforma una matriz de dimensiones
# (64, 64, 3) en un vector de 12.288 elementos.
#
# Se conserva la información numérica, pero se pierde
# la organización espacial explícita de la matriz.
vector_caracteristicas = imagen_normalizada.flatten()

print("Forma del vector:", vector_caracteristicas.shape)
print("Número de características:", vector_caracteristicas.size)

# ------------------------------------------------------------
# 5. Mostrar la categoría correspondiente
# ------------------------------------------------------------

print("Categoría:", dataset.classes[etiqueta])



## 07. Preparación de una muestra para el primer modelo

### Objetivo

Preparar 2.000 imágenes de entrenamiento para construir
un primer modelo de clasificación.

### ¿Por qué utilizamos una muestra?

El dataset de entrenamiento contiene 18.900 imágenes.

Para desarrollar y comprobar nuestro primer pipeline,
empezaremos con una muestra más pequeña.

Esto permite:

- Reducir el consumo de memoria.
- Acelerar los experimentos iniciales.
- Identificar errores antes de procesar todo el dataset.
- Comparar modelos de forma más eficiente.

### Decisiones técnicas

1. Seleccionaremos exclusivamente imágenes del conjunto
   de entrenamiento.

2. Utilizaremos muestreo estratificado para conservar
   aproximadamente las proporciones de clases.

3. Normalizaremos los valores RGB entre 0 y 1.

4. Convertiremos cada imagen en un vector de 12.288
   características.

5. Guardaremos las características en una matriz X y
   las etiquetas en un vector y.

El conjunto de validación y el de prueba permanecerán
separados.

### Convenciones de Machine Learning

- X: variables de entrada o características.
- y: variable objetivo o etiqueta que queremos predecir.
- X_train: características de entrenamiento.
- y_train: etiquetas de entrenamiento.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 07: Preparación de una muestra de entrenamiento
# ============================================================

import numpy as np
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1. Seleccionar 2.000 imágenes de entrenamiento
# ------------------------------------------------------------

# Partimos exclusivamente de indices_train.
# De esta forma no incorporamos imágenes de validación
# ni de prueba durante el entrenamiento.
#
# train_size=2000 selecciona exactamente 2.000 imágenes.
#
# random_state=42 permite reproducir la selección.
#
# stratify mantiene aproximadamente las proporciones
# originales de las diez categorías.

indices_muestra, _ = train_test_split(
    indices_train,
    train_size=2000,
    random_state=42,
    stratify=etiquetas[indices_train]
)

print("Imágenes seleccionadas:", len(indices_muestra))

# ------------------------------------------------------------
# 2. Crear una función de preparación de imágenes
# ------------------------------------------------------------

def preparar_imagen(indice):
    """
    Convierte una imagen de EuroSAT en un vector numérico.

    Parámetros
    ----------
    indice : int
        Posición de la imagen dentro del dataset.

    Devuelve
    --------
    vector : np.ndarray
        Vector de 12.288 características normalizadas.

    etiqueta : int
        Identificador numérico de la categoría.
    """

    # Recuperamos una imagen RGB y su etiqueta.
    imagen, etiqueta = dataset[int(indice)]

    # Convertimos la imagen a un array de NumPy.
    imagen_array = np.asarray(imagen)

    # Normalizamos los valores RGB a [0, 1].
    # float32 reduce el consumo de memoria frente a float64.
    imagen_normalizada = (
        imagen_array.astype(np.float32) / 255.0
    )

    # Convertimos la matriz (64, 64, 3)
    # en un vector de 12.288 características.
    vector = imagen_normalizada.flatten()

    return vector, etiqueta


# ------------------------------------------------------------
# 3. Procesar todas las imágenes de la muestra
# ------------------------------------------------------------

# Creamos dos listas:
# una para los vectores y otra para las etiquetas.
caracteristicas = []
etiquetas_muestra = []

for indice in indices_muestra:

    vector, etiqueta = preparar_imagen(indice)

    caracteristicas.append(vector)
    etiquetas_muestra.append(etiqueta)

# ------------------------------------------------------------
# 4. Convertir las listas en arrays
# ------------------------------------------------------------

# Cada fila de X representa una imagen.
# Cada columna representa una característica RGB.
X_train_sample = np.stack(caracteristicas)

# y contiene las etiquetas numéricas correspondientes.
y_train_sample = np.array(etiquetas_muestra)

# ------------------------------------------------------------
# 5. Comprobar las dimensiones
# ------------------------------------------------------------

print("Forma de X:", X_train_sample.shape)
print("Forma de y:", y_train_sample.shape)

print("Tipo de dato X:", X_train_sample.dtype)

print("Valor mínimo:", X_train_sample.min())
print("Valor máximo:", X_train_sample.max())

# Verificamos que haya una etiqueta por imagen.
assert len(X_train_sample) == len(y_train_sample)

# Verificamos que las imágenes tengan 12.288 características.
assert X_train_sample.shape[1] == 64 * 64 * 3

print("Muestra preparada correctamente.")


In [ ]:
# ============================================================
# ETAPA 07B: Preparar la muestra de validación RGB
# ============================================================

# Mantenemos la misma selección utilizada en el
# notebook original para comparar los experimentos.
# Seleccionamos 1.000 imágenes SOLO de validación.
indices_val_muestra, _ = train_test_split(
    indices_val,
    train_size=1000,
    random_state=42,
    stratify=etiquetas[indices_val]
)

# Aplicamos el mismo preprocesamiento RGB que en train.
# No utilizamos imágenes del conjunto test.
X_val_lista = []
y_val_lista = []

for indice in indices_val_muestra:
    vector, etiqueta = preparar_imagen(indice)
    X_val_lista.append(vector)
    y_val_lista.append(etiqueta)

X_val_sample = np.stack(X_val_lista)
y_val_sample = np.array(y_val_lista)

# Comprobaciones de coherencia: dimensiones y etiquetas.
assert X_val_sample.shape == (1000, 12288)
assert y_val_sample.shape == (1000,)
assert np.array_equal(y_val_sample, etiquetas[indices_val_muestra])

print("Validación RGB preparada:", X_val_sample.shape)
print("Etiquetas de validación:", y_val_sample.shape)



## 13. Experimento con características HOG

### Objetivo

Explorar una representación alternativa de las imágenes
mediante Histogram of Oriented Gradients (HOG).

### ¿Qué es HOG?

Es un descriptor que representa la distribución de
orientaciones de gradientes (cambios de intensidad)
en distintas regiones de una imagen.

Permite describir determinados patrones de bordes
y estructuras locales.

### Hipótesis

Una representación basada en estructuras visuales
podría generalizar mejor que los píxeles RGB aplanados.

### Limitaciones

HOG puede perder información de color importante
para diferenciar categorías de cobertura del suelo.

Evaluaremos esta hipótesis mediante métricas,
sin asumir de antemano una mejora.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 13: Comprobar disponibilidad de scikit-image
# ============================================================

# importlib.metadata permite consultar la versión
# instalada de una biblioteca de Python.
from importlib.metadata import version

# hog es la función que utilizaremos para extraer
# características de bordes y gradientes.
from skimage.feature import hog

# Comprobamos que la importación funciona.
print("scikit-image:", version("scikit-image"))

print("Función HOG importada correctamente.")



## 14. Extracción y visualización de características HOG

### Objetivo

Extraer un descriptor HOG de una imagen RGB de EuroSAT
y visualizar los patrones de gradientes que representa.

### Conceptos

**Gradiente:** cambio de intensidad entre píxeles vecinos.

**Orientación:** dirección en la que cambia la intensidad.

**HOG:** descriptor que resume las orientaciones de
los gradientes en pequeñas regiones de una imagen.

**Celda HOG:** región local utilizada para calcular
histogramas de orientaciones.

**Bloque HOG:** agrupación de celdas utilizada para
normalizar sus histogramas.

### Hipótesis

Las estructuras y texturas de algunas categorías
podrían distinguirse mejor mediante HOG que mediante
valores RGB aplanados.

### Limitación

Trabajaremos en escala de grises, por lo que perderemos
información de color que podría ser relevante para
diferenciar vegetación, agua y superficies urbanas.

Este paso es exploratorio: todavía no entrenamos
ningún modelo nuevo.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 14: Extracción y visualización de HOG
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from skimage.color import rgb2gray
from skimage.feature import hog
from skimage import exposure

# ------------------------------------------------------------
# 1. Recuperar una imagen del conjunto de entrenamiento
# ------------------------------------------------------------

# Reutilizamos el índice seleccionado anteriormente.
# Esta imagen pertenece exclusivamente a entrenamiento.
imagen_pil, etiqueta = dataset[int(indices_train[0])]

# Convertimos la imagen RGB a un array de NumPy.
imagen_rgb = np.asarray(imagen_pil)

# ------------------------------------------------------------
# 2. Convertir la imagen a escala de grises
# ------------------------------------------------------------

# HOG puede calcularse sobre imágenes en escala de grises.
#
# rgb2gray combina los canales RGB para obtener
# una imagen bidimensional de intensidades.
#
# La salida tiene dimensiones (64, 64).
imagen_gris = rgb2gray(imagen_rgb)

print("Forma RGB:", imagen_rgb.shape)
print("Forma escala de grises:", imagen_gris.shape)

# ------------------------------------------------------------
# 3. Extraer características HOG
# ------------------------------------------------------------

# orientations=9:
#   Utilizamos 9 intervalos de orientación.
#
# pixels_per_cell=(8, 8):
#   Cada celda HOG representa una región de 8×8 píxeles.
#
# cells_per_block=(2, 2):
#   Cada bloque agrupa 2×2 celdas para normalizar
#   los histogramas de gradientes.
#
# visualize=True:
#   Devuelve también una imagen que permite observar
#   una representación visual de los gradientes.
#
# feature_vector=True:
#   Devuelve las características como vector 1D.
#
# transform_sqrt=False:
#   Desactivamos la transformación de raíz cuadrada
#   para comenzar con una configuración sencilla.

vector_hog, imagen_hog = hog(
    imagen_gris,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
    feature_vector=True,
    transform_sqrt=False
)

# ------------------------------------------------------------
# 4. Mejorar la visualización HOG
# ------------------------------------------------------------

# RescaleIntensity ajusta el contraste de visualización.
#
# No modifica el vector de características que
# utilizaríamos posteriormente para entrenar.
imagen_hog_visual = exposure.rescale_intensity(
    imagen_hog,
    in_range=(0, np.percentile(imagen_hog, 99))
)

# ------------------------------------------------------------
# 5. Comparar la imagen original con HOG
# ------------------------------------------------------------

figura, ejes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(10, 5)
)

# Imagen RGB original.
ejes[0].imshow(imagen_rgb)
ejes[0].set_title("Imagen RGB original")
ejes[0].axis("off")

# Representación visual de HOG.
ejes[1].imshow(
    imagen_hog_visual,
    cmap="gray"
)
ejes[1].set_title("Visualización HOG")
ejes[1].axis("off")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Comprobar las dimensiones del descriptor
# ------------------------------------------------------------

print("Categoría:", dataset.classes[etiqueta])

print("Características RGB:", imagen_rgb.size)

print("Características HOG:", len(vector_hog))

print("Forma del vector HOG:", vector_hog.shape)



## 15. Preparación de características HOG

### Objetivo

Extraer características HOG de las mismas imágenes
utilizadas en nuestro experimento anterior.

Conjuntos:
- Entrenamiento: 2.000 imágenes.
- Validación: 1.000 imágenes.
- Prueba: permanece reservada.

### Hipótesis

Una representación basada en patrones locales
de gradientes podría mejorar la generalización
respecto a los píxeles RGB aplanados.

### Decisiones técnicas

1. Conservamos los mismos índices de imágenes.
2. Aplicamos idénticos parámetros HOG a ambos conjuntos.
3. No utilizamos datos de prueba.
4. No entrenamos ningún modelo todavía.

### Limitaciones

La conversión a escala de grises elimina información
de color, que podría ser relevante para distinguir
tipos de vegetación y masas de agua.

Una reducción en el número de características no
garantiza una mejora del rendimiento.


In [ ]:

# ============================================================
# PROYECTO: GeoAI Land Cover Classifier
# ETAPA 15: Preparación de características HOG
# ============================================================

import time
import numpy as np

from skimage.color import rgb2gray
from skimage.feature import hog

# ------------------------------------------------------------
# 1. Definir una función reutilizable
# ------------------------------------------------------------

def extraer_hog(indice):
    """
    Extrae características HOG de una imagen EuroSAT.

    Parámetros
    ----------
    indice : int
        Posición de la imagen dentro del dataset.

    Devuelve
    --------
    caracteristicas : np.ndarray
        Vector HOG de 1.764 características.

    etiqueta : int
        Categoría numérica de la imagen.
    """

    # Recuperamos la imagen y su etiqueta.
    imagen, etiqueta = dataset[int(indice)]

    # Convertimos PIL a un array RGB.
    imagen_rgb = np.asarray(imagen)

    # Convertimos a escala de grises.
    # Así aplicamos la misma configuración HOG
    # que utilizamos en el paso anterior.
    imagen_gris = rgb2gray(imagen_rgb)

    # Extraemos el descriptor HOG.
    # visualize=False evita calcular una imagen
    # adicional que no necesitamos para entrenar.
    caracteristicas = hog(
        imagen_gris,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=False,
        feature_vector=True,
        transform_sqrt=False
    )

    # Convertimos a float32 para reducir memoria.
    caracteristicas = caracteristicas.astype(np.float32)

    return caracteristicas, etiqueta


# ------------------------------------------------------------
# 2. Función para procesar múltiples imágenes
# ------------------------------------------------------------

def preparar_conjunto_hog(indices_conjunto):
    """
    Convierte una colección de imágenes en una matriz HOG.

    Devuelve
    --------
    X : matriz de características
    y : vector de etiquetas
    """

    vectores = []
    etiquetas_conjunto = []

    for indice in indices_conjunto:

        vector, etiqueta = extraer_hog(indice)

        vectores.append(vector)
        etiquetas_conjunto.append(etiqueta)

    # Cada fila representa una imagen.
    X = np.stack(vectores)

    # Cada elemento representa su clase real.
    y = np.array(etiquetas_conjunto)

    return X, y


# ------------------------------------------------------------
# 3. Procesar entrenamiento y validación
# ------------------------------------------------------------

inicio = time.time()

X_train_hog, y_train_hog = preparar_conjunto_hog(
    indices_muestra
)

X_val_hog, y_val_hog = preparar_conjunto_hog(
    indices_val_muestra
)

tiempo_extraccion = time.time() - inicio


# ------------------------------------------------------------
# 4. Comprobar resultados y correspondencia
# ------------------------------------------------------------

print("Forma X_train_hog:", X_train_hog.shape)
print("Forma y_train_hog:", y_train_hog.shape)

print("Forma X_val_hog:", X_val_hog.shape)
print("Forma y_val_hog:", y_val_hog.shape)

# Confirmamos que las etiquetas corresponden
# exactamente a las de los experimentos anteriores.
assert np.array_equal(y_train_hog, y_train_sample)
assert np.array_equal(y_val_hog, y_val_sample)

# Comprobamos el número de características.
assert X_train_hog.shape[1] == 1764
assert X_val_hog.shape[1] == 1764

# Verificamos que no haya valores NaN o infinitos.
assert np.isfinite(X_train_hog).all()
assert np.isfinite(X_val_hog).all()

print(
    "Tiempo de extracción:",
    round(tiempo_extraccion, 2),
    "segundos"
)

print("Matrices HOG preparadas correctamente.")


**Resultado:** matrices RGB y HOG de 2.000 imágenes de train y 1.000 de validación. Las variables son temporales: el notebook 03 las reconstruye al ejecutarse; no requiere importar este notebook. El split por imagen no garantiza independencia geográfica.